In [ ]:
# Sort simulations by their sim_xx key
import json
import re
from pathlib import Path

# Load data
# ############# Dach #############
# json_file = "./sim_results/MOE/DACH-VWS/DACH-VWS.json"  # DONE
# json_file = "./sim_results/MOE/DACH-VWS_6cores/DACH-VWS.json"

# ############# SideBeam #########
# json_file = "./sim_results/MOE/Laengstraeger/Laengstraeger_02.json"
# json_file = "./sim_results/MOE/Laengstraeger_softLimits/Laengstraeger_02.json"
# ############# SeatShell #########
# json_file = "./sim_results/MOE/SeatShell/SeatShell.json"
# json_file = "./sim_results/MLVGP/SeatShell_168/SeatShell.json"  # with 168 initial samples
json_file = "./sim_results/MLVGP/SeatShell_1/SeatShell.json"  # with 1 initial sample
# json_file = "./sim_results/MLVGP/SeatShell_168_highestpeak_added/SeatShell.json"
# json_file = "./sim_results/MOE/SeatShell_withpredicted/SeatShell.json"
#
# weather to save the figs or not
save = False
# save = True
# gpu_experts = False
gpu_experts = True
# selected_indices = None


json_file2 = None
with open(json_file) as f:
    data = json.load(f)
    # rename the keys
    data = {f"$sim_{{{int(k.split('_')[1])}}}$": v for k, v in data.items()}

# create save folder
######################
# Figures are saved relative to this notebook when save=True
target_dir = Path("./figures")
target_dir.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename, fmt="svg", **kwargs):
    """
    Save a matplotlib figure to the specified directory if saving is enabled.

    Args:
        fig (matplotlib.figure.Figure): Figure to save.
        filename (str): Name of the file (e.g., "figure.svg").
        fmt (str): File format (default "svg").
        **kwargs: Extra arguments passed directly to fig.savefig().
    """
    if save:
        # Pass all extra keyword arguments, including bbox_inches='tight'
        fig.savefig(target_dir / filename, format=fmt, **kwargs)
        print(f"Saved figure to {target_dir / filename}")

## Load Plotting Functions


In [ ]:
# merge data if desired:
######################
# json_file2 = "./sim_results/MOE/DACH-VWS/DACH-VWS.json"
# json_file2 = "./sim_results/MOE/SeatShell_withpredicted/SeatShell.json"
# # # Load the second dataset (same or another file)
# with open(json_file2, "r") as f:
#     data2 = json.load(f)
#     data2
#     {f"$sim_{{{int(k.split('_')[1])}}}$": v for k, v in data2.items()}
# existing_indices = [
#     int(re.search(r"sim_(\d+)", k).group(1))
#     for k in data.keys()
#     if re.match(r"sim_\d+", k)
# ]
# next_index = max(existing_indices, default=0) + 1
# renamed_data = {f"sim_{i + next_index:02d}": v for i, (k, v) in enumerate(data2.items())}
# print(data.keys())
# print(renamed_data.keys())
# data.update(renamed_data)

In [ ]:
# Expert
from ensima.helpers.co2 import estimate_co2
from ensima.helpers.energy import estimate_energy

if gpu_experts:
    # merge experts:
    ######################
    if "DACH" in json_file:
        # json_file2 = "./sim_results/no_optimization/DACH-VWS_no_optimization.json"
        # mode = ["expert", "user"]
        json_file2 = "./sim_results/no_optimization/DACH-VWS_no_optimization_expert_only.json"
        mode = ["expert"]
    elif "Laengstraeger_02" in json_file:
        json_file2 = "./sim_results/no_optimization/Laengstraeger_02_no_optimization.json"
        mode = ["expert"]
    elif "SeatShell" in json_file:
        json_file2 = "./sim_results/no_optimization/SeatShell_no_optimization.json"
        mode = ["expert"]
    with open(json_file2) as f:
        data2 = json.load(f)
    # Replace keys with 'mode' and store in a new dict called 'exper'
    experts = {m: v for m, (_, v) in zip(mode, data2.items())}
    print("exper keys:", experts.keys())
else:
    if "DACH" in json_file:
        experts = {
            # "unskilled user": {
            "user": {
                "jobname": "DACH",
                "x": [
                    [140.00, 0.70, 3.00, 0.00, 0.00, 6.4, 6.9],
                    [140.00, 0.70, 3.00, 0.00, 220.00, 11.1, 2.8],
                    [140.00, 0.70, 3.00, 0.00, 800.00, 21.3, 1.5],
                    [140.00, 0.70, 3.00, 0.06, 800.00, 24.2, 0.9],
                    [140.00, 0.70, 6.00, 0.06, 800.00, 24.7, 0.6],
                    [140.00, 0.70, 9.00, 0.06, 800.00, 26.0, 0.5],
                    [140.00, 0.70, 9.00, 0.06, 220.00, 20.8, 0.4]
                ],
                "y": [
                    [38.9, 34.40, 15.90, 10.80, 0.00, 0.00, 0.00],
                    [14.4, 13.20, 10.50, 61.90, 0.00, 0.00, 0.00],
                    [1.9, 10.00, 2.80, 85.30, 0.00, 0.00, 0.00],
                    [1.9, 9.80, 2.50, 85.80, 0.00, 0.00, 0.00],
                    [1.8, 9.10, 3.20, 85.90, 0.00, 0.00, 0.00],
                    [1.4, 4.10, 7.90, 86.60, 0.00, 0.00, 0.00],
                    [0.5, 2.60, 9.10, 87.80, 0.00, 0.00, 0.00]
                ],
                "best_output": [0.5, 2.60, 9.10, 87.80, 0.00, 0.00, 0.00],
                "elapsed_seconds": [
                    2 * 3600 + 18 * 60 + 12,
                    3 * 3600 + 5 * 60 + 39,
                    2 * 3600 + 36 * 60 + 50,
                    2 * 3600 + 30 * 60 + 25,
                    3 * 3600 + 6 * 60 + 36,
                    3 * 3600 + 21 * 60 + 59,
                    2 * 3600 + 48 * 60 + 17
                ],
                "args": {
                    "x_fields": ["Rp", "D", "p", "Fr", "db", "dD", "dT"],
                    "y_fields": ["L1", "L2", "L3", "L4", "L5", "L6", "L7"]
                }
            },
            "expert": {
                "jobname": "DACH",
                "x": [
                    [140.00, 0.70, 3.00, 0.00, 0.00, 6.4, 6.9],
                    [140.00, 0.70, 3.00, 0.00, 220.00, 11.1, 2.8],
                    [140.00, 0.70, 6.00, 0.03, 220.00, 16.2, 0.9],
                    [140.00, 0.70, 9.00, 0.06, 220.00, 20.8, 0.4],
                ],
                "y": [
                    [38.9, 34.40, 15.90, 10.80, 0.00, 0.00, 0.00],
                    [14.4, 13.20, 10.50, 61.90, 0.00, 0.00, 0.00],
                    [0.9, 9.90, 7.70, 81.50, 0.00, 0.00, 0.00],
                    [0.5, 2.60, 9.10, 87.80, 0.00, 0.00, 0.00],
                ],
                "best_output": [0.5, 2.60, 9.10, 87.80, 0.00, 0.00, 0.00],
                "elapsed_seconds": [
                    2 * 3600 + 18 * 60 + 12,  # 8292
                    3 * 3600 + 5 * 60 + 39,  # 11139
                    2 * 3600 + 49 * 60 + 5,  # 10145
                    2 * 3600 + 48 * 60 + 17,  # 10117
                ],
                "args": {
                    "x_fields": ["Rp", "D", "p", "Fr", "db", "dD", "dT"],
                    "y_fields": ["L1", "L2", "L3", "L4", "L5", "L6", "L7"],
                },
            },
        }

    elif "Laengstraeger_02" in json_file:
        experts = {
            "expert": {
                "jobname": "SideBeam",
                "x": [
                    [597.00, 1.20, 3.00, 0.03, 0.00, 34.8, 18.6],
                    [597.00, 1.00, 3.00, 0.12, 0.00, 27.4, 14.9],
                    [390.00, 1.00, 3.00, 0.12, 0.00, 26.8, 15.9],
                    [390.00, 1.00, 2.00, 0.12, 0.00, 25.2, 14.9],
                ],
                "y": [
                    [4.4, 26.00, 11.30, 58.20, 0.00, 0.00, 0.00],
                    [8.7, 21.00, 11.60, 58.70, 0.00, 0.00, 0.00],
                    [6.3, 19.90, 10.90, 62.90, 0.00, 0.00, 0.00],
                    [6.9, 20.80, 12.40, 59.90, 0.00, 0.00, 0.00],
                ],
                "best_output": [6.9, 20.80, 12.40, 59.90, 0.00, 0.00, 0.00],
                "elapsed_seconds": [
                    1 * 3600 + 46 * 60 + 20,  # 6380
                    1 * 3600 + 4 * 60 + 0,  # 3840
                    1 * 3600 + 42 * 60 + 54,  # 6174
                    1 * 3600 + 44 * 60 + 30,  # 6270
                ],
                "args": {
                    "x_fields": ["Rp", "D", "p", "Fr", "db", "dD", "dT"],
                    "y_fields": ["L1", "L2", "L3", "L4", "L5", "L6", "L7"],
                },
            },
        }

    # Repeat the same pattern for SeatShell or other parts
    elif "SeatShell" in json_file:
        experts = {
            "expert": {
                "jobname": "SeatShell",
                "x": [
                    [250.00, 1.10, 3.00, 0.03, 0.00, 35.6, 4.80],
                    [250.00, 1.10, 3.00, 0.03, 800.00, 31.6, 5.20],
                    [250.00, 1.10, 6.00, 0.03, 800.00, 31.8, 3.90],
                    [250.00, 1.10, 9.00, 0.03, 800.00, 29.5, 3.20],
                ],
                "y": [
                    [0.2, 17.40, 26.30, 56.10, 0.00, 0.00, 0.00],
                    [0.3, 10.50, 20.00, 69.20, 0.00, 0.00, 0.00],
                    [0.3, 8.20, 18.30, 73.20, 0.00, 0.00, 0.00],
                    [0.1, 7.90, 15.20, 76.80, 0.00, 0.00, 0.00],
                ],
                "best_output": [0.1, 7.90, 15.20, 76.80, 0.00, 0.00, 0.00],
                "elapsed_seconds": [
                    1 * 3600 + 55 * 60 + 45,  # 6945
                    1 * 3600 + 55 * 60 + 45,  # 6945
                    1 * 3600 + 42 * 60 + 54,  # 6174
                    1 * 3600 + 44 * 60 + 30,  # 6270
                ],
                "args": {
                    "x_fields": ["Rp", "D", "p", "Fr", "db", "dD", "dT"],
                    "y_fields": ["L1", "L2", "L3", "L4", "L5", "L6", "L7"],
                },
            },
        }

    if len(experts) > 0:
        if "DACH" in json_file:
            cores = 6
            cpu_frequency_Hz = 1.596e9
        elif "Laengstraeger" in json_file or "SeatShell.json" in json_file:
            cores = 4
            cpu_frequency_Hz = 2.801e9

        for sim in experts.values():
            sim["energy"] = [estimate_energy(t, cores, frequency=cpu_frequency_Hz) for t in sim["elapsed_seconds"]]
            sim["co2"] = [estimate_co2(e) for e in sim["energy"]]
            sim["progress"] = [100 for e in sim["energy"]]

    sep = "\\ "
    experts = {f"${k.replace(' ', sep)}$": v for k, v in experts.items()}


In [ ]:
# %%
# plot_simulations.py (Jupyter-compatible)
import os

import numpy as np

# Force Qt to use X11 (xcb) instead of Wayland
os.environ["QT_QPA_PLATFORM"] = "xcb"
# Force software OpenGL rendering (avoid AMDGPU issues)
os.environ["QT_OPENGL"] = "software"
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"
import matplotlib

matplotlib.use("QtAgg")
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.ticker import MaxNLocator

# Inline plotting if run in Jupyter
%matplotlib inline
# %matplotlib widget
# %matplotlib notebook
# %matplotlib qt
# %matplotlib tk

# Sort simulations by their sim_xx key
sims = sorted(data.keys())


def set_axis_fontsize(ax, label_size=14, tick_size=12, legend_size=12):
    """
    Set font size for axis labels, tick labels, and legend.

    Parameters:
        ax (matplotlib.axes.Axes): The axes object to modify.
        label_size (int): Font size for x and y labels.
        tick_size (int): Font size for x and y ticks.
        legend_size (int): Font size for the legend.
    """
    ax.xaxis.label.set_size(label_size)
    ax.yaxis.label.set_size(label_size)
    ax.tick_params(axis='x', labelsize=tick_size)
    ax.tick_params(axis='y', labelsize=tick_size)

    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_fontsize(legend_size)


def plot_xy(x_values, y_arrays, ylabel="Value", xlabel="Simulation", title="Plot"):
    """
    Plot values per simulation with connected lines across iterations.
    Line color indicates iteration. Global best (minimum) is highlighted.

    Args:
        x_values (list): X-axis values (simulation names or indices)
        y_arrays (list of list/np.ndarray): Each inner list contains iteration values for that simulation
        ylabel (str): Y-axis label
        title (str): Plot title
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    n_sims = len(x_values)
    max_iter = max(len(arr) for arr in y_arrays)

    # Colormap for iterations
    cmap = plt.get_cmap("viridis", max_iter)

    # Plot lines per iteration
    for iter_idx in range(max_iter):
        line_y = []
        line_x = []
        for i, y_list in enumerate(y_arrays):
            if iter_idx < len(y_list):
                line_y.append(y_list[iter_idx])
                line_x.append(i)
        if line_y:
            plt.plot(line_x, line_y, lw=2, color=cmap(iter_idx), label=f"Iteration {iter_idx + 1}")

    # Highlight global best (minimum)
    all_y = []
    for y_list in y_arrays:
        all_y.extend(y_list)
    all_y = np.array(all_y, dtype=float)
    np.nanargmin(all_y)

    # cum_len = 0
    # for i, y_list in enumerate(y_arrays):
    #     y = np.array(y_list, dtype=float)
    #     if best_idx_flat < cum_len + len(y):
    #         local_idx = best_idx_flat - cum_len
    #         plt.scatter(i, y[local_idx], color="red", s=120, edgecolors="black", zorder=10,
    #                     label=f"Global Best: {y[local_idx]:.3f}")
    #         break
    #     cum_len += len(y)

    plt.xticks(ticks=np.arange(n_sims), labels=x_values)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    # plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    set_axis_fontsize(ax)
    plt.tight_layout()
    plt.show()


def scatter_xy(x_values, y_arrays, ylabel="Value", xlabel="Simulation", title="Scatter Plot", show=True):
    """
    Scatter plot where each x_value can have multiple y-values.

    Args:
        x_values (list): X-axis values (e.g., simulation names or indices)
        y_arrays (list of list/np.ndarray): Each inner list contains Y-values for that X
        ylabel (str): Y-axis label
        title (str): Plot title
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    for i, y_list in enumerate(y_arrays):
        y = np.array(y_list, dtype=float)
        x = np.full_like(y, i, dtype=float)  # same x for all points in this sim
        plt.scatter(x, y, s=80, label=str(x_values[i]))

    plt.xticks(ticks=np.arange(len(x_values)), labels=x_values)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    set_axis_fontsize(ax)
    plt.tight_layout()
    if show:
        plt.show()

    return fig, ax


def scatter_xy_axes(x_values, y_arrays, ylabel="Value", xlabel="Simulation", title="Scatter Plot"):
    fig, ax = scatter_xy(x_values, y_arrays, ylabel=ylabel, xlabel=xlabel, title=title, show=False)
    return fig, ax


def get_unit(value: float, unit_type: str) -> tuple[float, str]:
    """
    Format a numeric value with an automatically chosen unit based on its magnitude.

    Supports milli- and micro-units for small values.

    Args:
        value: The numeric value to be formatted.
        unit_type: The base unit type, either "J" for energy or "g" for mass.

    Returns:
        factor,unit

    Raises:
        ValueError: If the unit_type is not supported.
    """
    thresholds = {
        "J": [("MJ", 1e6), ("kJ", 1e3), ("J", 1), ("mJ", 1e-3), ("µJ", 1e-6)],
        "g": [("t", 1e6), ("kg", 1e3), ("g", 1), ("mg", 1e-3), ("µg", 1e-6)],
    }

    if unit_type not in thresholds:
        raise ValueError(f"Unsupported unit type: {unit_type}")

    abs_value = abs(value)  # Handle negative values gracefully

    for unit, factor in thresholds[unit_type]:
        if abs_value >= factor:
            return factor, unit

    # If value is extremely small
    return (
        1,
        unit_type,
    )


def highlight_best_points(ax, y, sims, y_idx, best_points, x=None):
    """
    Highlight top points on a plot with gradient colors.

    Args:
        ax : matplotlib axis
            Axis on which to plot the points.
        y : list of np.ndarray
            Simulation outputs, each element is (n_iters, n_y_fields).
        sims : list of str
            Names of the simulations, in the same order as y.
        y_idx : int
            Index of the y field to plot.
        best_points : list of dict
            Each dict contains keys 'sim' and 'iter'.
        x : list of np.ndarray or None, optional
            Optional x-values for each simulation. Must match shape of y.
            If None, iteration indices are used.
    """
    colors = plt.cm.cool(np.linspace(0, 1, len(best_points)))  # Red → Blue gradient
    # colors = plt.cm.viridis(np.linspace(0, 1, len(best_points)))

    for rank, (info, color) in enumerate(zip(best_points, colors), 1):
        sim_idx = sims.index(info["sim"])
        iter_idx = info["iter"]
        y_val = float(y[sim_idx][iter_idx, y_idx])

        # Use provided x if available, else iteration index
        x_val = x[sim_idx][iter_idx] if x is not None else iter_idx
        ax.scatter(
            x_val, y_val,
            s=120, edgecolors="black",
            linewidths=1.5, color=color,
            zorder=5, label=f"Top {rank}"
        )
        # ax.scatter(
        #     x_val, y_val,
        #     s=120,  # marker size
        #     facecolor='none',  # empty inside
        #     edgecolor=color,  # edge color
        #     linewidths=3.0,  # thicker edge
        #     zorder=5,
        #     label=f"Top {rank}"
        # )
        info["rank"] = f"Top {rank}"

In [ ]:
from ensima.helpers.energy import estimate_energy

# Fixed CPU frequency in Hz
# max
cpu_frequency_Hz_max = 4315.625 * 1e6
#min:
cpu_frequency_Hz_min = 1500 * 1e6

# cpu_frequency_Hz = (cpu_frequency_Hz_min + cpu_frequency_Hz_max) / 2
cpu_frequency_Hz = cpu_frequency_Hz_min

# Collect elapsed times
elapsed_seconds = [data[k].get("elapsed_seconds", None) for k in data]

# Compute energy
energy = []
for i, sim_times in enumerate(elapsed_seconds):
    cores = data[sims[i]].get("args")["cores"]
    sim_energy = [estimate_energy(t, cores, frequency=cpu_frequency_Hz) for t in sim_times]
    energy.append(sim_energy)
    # Replace in the data dict
    data[sims[i]]["energy"] = sim_energy

if len(experts) and gpu_experts > 0:
    energy_exper = []
    for mode_name, sim in experts.items():
        cores = sim.get("args", {}).get("cores")  # default to 1 if missing
        sim_times = sim.get("elapsed_seconds", [])
        sim_energy = [estimate_energy(t, cores, frequency=cpu_frequency_Hz) for t in sim_times]
        energy_exper.append(sim_energy)
        experts[mode_name]["energy"] = sim_energy

In [ ]:
# energy = [data[k].get("energy") or data[k].get("sim_energy", None) for k in data]
# sims = sorted(data.keys())
# # print(energy)
# # print(sims)
# _ = plot_xy(sims, energy, ylabel="Energy", title="Energy per Simulation")


In [ ]:
energy = [data[k].get("energy") or data[k].get("sim_energy", None) for k in data]
total_time = [data[k].get("total_time_in_seconds") for k in data]
sims = sorted(data.keys())
if experts:
    sims_with_expert = sims + list(experts.keys())
    energy_with_expert = energy + [sim["energy"] for sim in experts.values()]
    # energy_with_expert = energy + [sim.get("energy") or sim.get("sim_energy", None) for sim in experts.values()]
    fig, ax = scatter_xy(sims_with_expert, energy_with_expert, ylabel="Energy", title="Energy per Simulation")
else:
    fig, ax = scatter_xy(sims, energy, ylabel="Energy", title="Energy per Simulation")

save_figure(fig, "energy_per_sim.svg", fmt="svg")

fig, ax = scatter_xy(sims, total_time, ylabel="Total Time (s)", title="")
save_figure(fig, "total_time_per_sim.svg", fmt="svg")



In [ ]:
best_output = np.array([data[k].get("best_output", None) for k in data])
# y_fields = data[sims[-1]]["args"]["y_fields"]
# sims = sorted(data.keys())
# # print(best_output)
# # print(y_fields)
# index = 1
# values = [[float(n)] for n in best_output[:, index]]
# y_field = y_fields[index]
# plot_xy(sims, values, ylabel="Best Output", title=f"{y_field}")

In [ ]:
y_fields = data[sims[-1]]["args"]["y_fields"]
best_output = np.array([data[k].get("best_output", None) for k in data])
if experts:
    best_output = np.concatenate(
        (best_output, np.array([sim["best_output"] for sim in experts.values()])),
        axis=0
    )

for index in range(0, 7):
    values = [[float(n)] for n in best_output[:, index]]
    y_field = y_fields[index]
    if experts:
        _ = plot_xy(sims_with_expert, values, ylabel=f"Best {y_field}", title=f"{y_field}")
    else:
        _ = plot_xy(sims, values, ylabel=f"Best {y_field}", title=f"{y_field}")


Plot all Y- values versues the simulation

In [ ]:
from ensima.classes.logger import Logger
from ensima.helpers.optimum import find_min_or_max

# Latest args from the last simulation
latest_sim_name = sorted(data.keys())[-1]


class Args:
    target = data[latest_sim_name]["args"]["target"]
    attention_coefficients = data[latest_sim_name]["args"]["attention_coefficients"]


args = Args()

# Sorted simulations
sims = sorted(data.keys())
y = [np.array(data[k].get("y", None), dtype=float) for k in sims]
x = [np.array(data[k].get("x", None), dtype=float) for k in sims]
energy = [data[k].get("energy") or data[k].get("sim_energy", None) for k in data]
if experts:
    sims_with_expert = sims + list(experts.keys())
    y = y + [sim["y"] for sim in experts.values()]
    x = x + [sim["x"] for sim in experts.values()]
    energy = energy + [sim["energy"] for sim in experts.values()]

# Scale all y values with latest attention coefficients
y_scaled = [y_vals * np.array(args.attention_coefficients, dtype=float) for y_vals in y]

# Flatten all simulations into a single array
all_y_scaled = np.concatenate(y_scaled, axis=0)
x_dummy = np.zeros((all_y_scaled.shape[0], 1))  # placeholder

# Find top N = number of simulations
top_n = 3  #len(sims)
best_indices = []

# Work with a copy to avoid selecting the same row multiple times
logger = Logger(__name__, level="DEBUG").get()
# args.target = [0.0, 10.0, 50.0, 100.0, 50.0, 0.0, 0.0]
# args.attention_coefficients = [1, 0.01, 0.01, -0.01, 0.01, 1, 1]
y_copy = all_y_scaled.copy()
for _ in range(top_n):
    idx_best = find_min_or_max(y=y_copy, x=x_dummy, args=args, logger=logger, mode="minimization")
    best_indices.append(idx_best)
    # Mask the selected row so it won't be picked again
    y_copy[idx_best, :] = np.inf

# Map global indices back to sim and iteration
sim_lengths = [len(y_vals) for y_vals in y_scaled]
sim_boundaries = np.cumsum([0] + sim_lengths)

best_points = []
for idx_global in best_indices:
    sim_idx = np.searchsorted(sim_boundaries, idx_global, side="right") - 1
    iter_idx = idx_global - sim_boundaries[sim_idx]
    best_points.append({
        "sim": sims[sim_idx] if not experts else sims_with_expert[sim_idx],
        "name": sims[sim_idx] if not experts else sims_with_expert[sim_idx],
        "iter": iter_idx,
        "y": y[sim_idx],
        "y_scaled": y_scaled[sim_idx][iter_idx],
        "y_best": y[sim_idx][iter_idx],
        "x_best": x[sim_idx][iter_idx],
        "energy": energy[sim_idx][iter_idx]  # if energy exists
    })

# Print results
for i, bp in enumerate(best_points):
    print(f"\nTop {i + 1} point:")
    print("Simulation:", bp["sim"])
    print("Iteration:", bp["iter"])
    print("y_scaled:", bp["y_scaled"])
    print("y:", bp["y_best"])
    print("x:", bp["x_best"])

print(data[latest_sim_name]["args"]["x_fields"])


In [ ]:
# selected_indices = [0, 2, 4]
selected_indices = None

# Original full list of sims
all_sims = sorted(data.keys())

# Apply selected_indices if provided
if selected_indices is not None:
    sims = [all_sims[i] for i in selected_indices]
else:
    sims = all_sims.copy()

# Load y-values
y = [np.array(data[k].get("y", None), dtype=float) for k in sims]
y_fields = data[sims[-1]]["args"]["y_fields"]

# Add experts if any
if experts:
    expert_names = list(experts.keys())
    sims += expert_names
    y += [np.array(experts[name]["y"], dtype=float) for name in expert_names]

# --- Colors and markers ---
all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
markers = [all_markers[i % len(all_markers)] for i in range(len(sims))]

# Map colors based on original index (so selected_indices align)
tab10 = colormaps["tab10"]
colors = []
for sim in sims:
    if sim in all_sims:
        orig_idx = all_sims.index(sim)
    else:  # expert
        orig_idx = len(all_sims)  # pick next color
    colors.append(tab10(orig_idx % 10))

for idx, y_field in enumerate(y_fields):
    fig, ax = plt.subplots(figsize=(8, 3))

    # Plot all simulations
    for sim_idx, sim_name in enumerate(sims):
        y_vals = y[sim_idx][:, idx].astype(float)
        x_sim = np.full_like(y_vals, sim_idx, dtype=float)
        ax.plot(x_sim, y_vals,
                marker=markers[sim_idx],
                linestyle="-",
                color=colors[sim_idx],
                label=sim_name)

    # Highlight top points
    if best_points:
        top_colors = tab10(np.arange(len(best_points)) % 10)
        for rank, info in enumerate(best_points, 1):
            sim_name = info.get("sim")
            iter_idx = info.get("iter")
            if sim_name not in sims or iter_idx is None:
                continue
            sim_idx = sims.index(sim_name)
            y_val = y[sim_idx][iter_idx, idx]
            if np.isnan(y_val):
                continue
            ax.scatter(sim_idx, y_val,
                       s=120,
                       edgecolors="black",
                       linewidths=2,
                       marker=markers[sim_idx],
                       color=top_colors[rank - 1],
                       alpha=0.8,
                       zorder=6,
                       label=f"Top {rank}")
            info["rank"] = f"Top {rank}"

    ax.set_xticks(np.arange(len(sims)))
    ax.set_xticklabels(sims)
    ax.set_xlabel("Simulation")
    ax.set_ylabel(y_field)
    ax.set_title(f"{y_field} per Simulation")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(loc="best")
    set_axis_fontsize(ax)
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Prepare sims, x, y, fields ---
sims = sorted(data.keys())
y_fields = data[sims[-1]]["args"]["y_fields"]
x_fields = data[sims[-1]]["args"]["x_fields"]

# Initialize x and y arrays as object arrays
x = np.array([np.array(data[k].get("x", []), dtype=float) for k in sims], dtype=object)
y = np.array([np.array(data[k].get("y", []), dtype=float) for k in sims], dtype=object)

# --- Convert x and y to 3D arrays (num_sims, n_points, n_features) ---
n_points_x = max(arr.shape[0] for arr in x)
n_points_y = max(arr.shape[0] for arr in y)
n_x_fields = len(x_fields)
n_y_fields = len(y_fields)

x_full = np.full((len(x), n_points_x, n_x_fields), np.nan, dtype=float)
y_full = np.full((len(y), n_points_y, n_y_fields), np.nan, dtype=float)

for i, arr in enumerate(x):
    n_rows = min(arr.shape[0], n_points_x)
    n_cols = min(arr.shape[1], n_x_fields)
    x_full[i, :n_rows, :n_cols] = arr[:n_rows, :n_cols]

for i, arr in enumerate(y):
    n_rows = min(arr.shape[0], n_points_y)
    n_cols = min(arr.shape[1], n_y_fields)
    y_full[i, :n_rows, :n_cols] = arr[:n_rows, :n_cols]

x = x_full
y = y_full
# --- Add experts ---
if experts:
    for name, sim in experts.items():
        sims.append(name)

        # Prepare expert x
        expert_x = np.array(sim["x"], dtype=float)
        expert_x_full = np.full((n_points_x, len(x_fields)), np.nan)
        for i, f in enumerate(x_fields):
            if f in sim["args"]["x_fields"]:
                idx = sim["args"]["x_fields"].index(f)
                n_rows = min(expert_x.shape[0], n_points_x)
                expert_x_full[:n_rows, i] = expert_x[:n_rows, idx]
        x = np.concatenate((x, expert_x_full[np.newaxis, :, :]), axis=0)

        # Prepare expert y
        expert_y = np.array(sim["y"], dtype=float)
        expert_y_full = np.full((n_points_y, len(y_fields)), np.nan)
        n_rows = min(expert_y.shape[0], n_points_y)
        expert_y_full[:n_rows, :] = expert_y[:n_rows, :]
        y = np.concatenate((y, expert_y_full[np.newaxis, :, :]), axis=0)

# --- Plot configuration ---
full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
n_sims = len(sims)

# --- Group x dims plotting ---
for y_idx, y_field in enumerate(y_fields):
    n_x = len(x_fields)
    fig, axes = plt.subplots(1, n_x, figsize=(3.5 * n_x, 3.5), squeeze=False)

    for x_idx, x_field in enumerate(x_fields):
        ax = axes[0, x_idx]
        for sim_idx, sim_name in enumerate(sims):
            x_vals = x[sim_idx][:, x_idx].astype(float)
            y_vals = y[sim_idx][:, y_idx].astype(float)
            ax.scatter(
                x_vals,
                y_vals,
                marker=all_markers[sim_idx % len(all_markers)],
                color=full_colors[sim_idx % len(full_colors)],
                alpha=0.7,
                label=f"{sim_name}"
            )

        ax.set_xlabel(x_field)
        ax.set_ylabel(f"{y_field} (%)")
        ax.grid(True, linestyle="--", alpha=0.6)
        # ax.set_title(f"{y_field} vs {x_field}")
        ax.legend(loc="best")
        set_axis_fontsize(ax, legend_size=10)

    plt.tight_layout()
    plt.show()
    save_figure(fig, f"inputs_vs_{y_field}.svg", fmt="svg")


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

%matplotlib inline

# --- Configuration ---
selected_indices = None  # None = all simulations
predicted = False  # True to plot predicted ± std

# --- Prepare basic info ---
sims = sorted(data.keys())
y_fields = data[sims[-1]]["args"]["y_fields"]
n_y_fields = len(y_fields)

# Include experts
total_sims = len(sims) + len(experts)
# Determine max number of points across all simulations and experts
max_points_sims = max(len(data[k]["y"]) for k in sims if data[k].get("y") is not None)
max_points_experts = max(len(sim["y"]) for sim in experts.values()) if experts else 0
n_points_y = max(max_points_sims, max_points_experts)

# --- Pre-allocate arrays ---
y = np.full((total_sims, n_points_y, n_y_fields), np.nan, dtype=float)
y_pred = np.full_like(y, np.nan)
y_std = np.full_like(y, np.nan)

# --- Fill original simulations ---
for sim_idx, k in enumerate(sims):
    y_vals = np.array(data[k]["y"], dtype=float)
    y[sim_idx, :y_vals.shape[0], :] = y_vals

    if data[k].get("y_model_mean") is not None:
        y_pred_vals = np.array(data[k]["y_model_mean"], dtype=float)
        y_pred[sim_idx, :y_pred_vals.shape[0], :] = y_pred_vals

    if data[k].get("y_model_std") is not None:
        y_std_vals = np.array(data[k]["y_model_std"], dtype=float)
        y_std[sim_idx, :y_std_vals.shape[0], :] = y_std_vals

# --- Fill experts ---
for i, (name, sim) in enumerate(experts.items()):
    idx = len(sims) + i
    expert_y = np.array(sim["y"], dtype=float)
    if expert_y.shape[0] < n_points_y:
        expert_y = np.pad(expert_y, ((0, n_points_y - expert_y.shape[0]), (0, 0)), constant_values=np.nan)
    y[idx] = expert_y

    if sim.get("y_model_mean") is not None:
        expert_y_pred = np.array(sim["y_model_mean"], dtype=float)
        if expert_y_pred.shape[0] < n_points_y:
            expert_y_pred = np.pad(expert_y_pred, ((0, n_points_y - expert_y_pred.shape[0]), (0, 0)),
                                   constant_values=np.nan)
        y_pred[idx] = expert_y_pred

    if sim.get("y_model_std") is not None:
        expert_y_std = np.array(sim["y_model_std"], dtype=float)
        if expert_y_std.shape[0] < n_points_y:
            expert_y_std = np.pad(expert_y_std, ((0, n_points_y - expert_y_std.shape[0]), (0, 0)),
                                  constant_values=np.nan)
        y_std[idx] = expert_y_std

# --- Update sims list to include experts ---
sims += list(experts.keys())

# --- Plot indices ---
plot_indices = selected_indices if selected_indices is not None else list(range(len(sims)))

# --- Colors, markers, linestyles ---
full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
linestyles = ["-"] * len(plot_indices)
colors = [full_colors[i % len(full_colors)] for i in plot_indices]
markers = [all_markers[i % len(all_markers)] for i in plot_indices]

# --- Plot all y_fields ---
for y_idx, y_field in enumerate(y_fields):
    fig, ax = plt.subplots(figsize=(8, 3))
    iterations = np.arange(n_points_y)

    for plot_i, sim_idx in enumerate(plot_indices):
        sim_name = sims[sim_idx]

        # Actual values
        y_vals = y[sim_idx][:, y_idx]
        mask_actual = ~np.isnan(y_vals)
        ax.plot(
            iterations[mask_actual],
            y_vals[mask_actual],
            marker=markers[plot_i],
            linestyle=linestyles[plot_i],
            color=colors[plot_i],
            alpha=0.8,
            label=f"{sim_name}"
        )

        # Predicted ± std
        if predicted:
            mu = y_pred[sim_idx][:, y_idx]
            sigma = y_std[sim_idx][:, y_idx]
            mask_pred = ~np.isnan(mu)
            if np.any(mask_pred):
                if np.any(~np.isnan(sigma)):
                    ax.fill_between(
                        iterations[mask_pred],
                        mu[mask_pred] - sigma[mask_pred],
                        mu[mask_pred] + sigma[mask_pred],
                        color=colors[plot_i],
                        alpha=0.2
                    )
                ax.plot(
                    iterations[mask_pred],
                    mu[mask_pred],
                    linestyle="--",
                    color=colors[plot_i],
                    alpha=0.8,
                    label=f"{sim_name} $predicted$"
                )

        ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    plt.xlabel("iterations")
    plt.ylabel(f"{y_field} (%)")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(loc="upper right")
    # plt.tight_layout()
    # plt.tight_layout()
    fig.tight_layout(pad=1.5)
    set_axis_fontsize(ax)
    save_figure(fig, f"{y_field}_vs_iterations.svg", fmt="svg")
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FormatStrFormatter, MaxNLocator

# --- Configuration ---
selected_indices = None  # which simulations to plot
selected_y_fields = None  # which outputs to plot (None = all)
# selected_y_fields = ["L1", "L2", "L3", "L4", "L5", "L6", "L7", ]  # which outputs to plot (None = all)
normalize = False  # normalize values between 0 and 1
figsize_single = (4.5, 3)  # size for single plot
subplots_in_row = 2  # number of subplots per row if using subfigures
use_subfigures = True  # set True to plot multiple outputs in subplots
zoom_around_points = False  # new option
zoom_margin = 0.2  # 20% around the data
start_iter = None
if "DACH" in json_file:
    start_iter = [0, 4, 7]  # compute accuracy only after this iteration
    selected_y_fields = ["L1", "L2", "L4", "L6", "L7", ]  # which outputs to plot (None = all)


def plot_grouped_bars(values_dict, ylabel, title, filename, y_limit=None,
                      legend_loc="best", group_gap=0.5, legend_outside=False, zoom_window=None):
    """
    values_dict: dict[y_field] -> list/array of length len(sim_start_labels)
    group_gap: extra space between groups
    legend_outside: if True, place legend outside the plot on the right
    """
    # Consistency check
    n_groups = len(sim_start_labels)
    for y_field, vals in values_dict.items():
        if len(vals) != n_groups:
            raise ValueError(f"Length mismatch for {y_field}: {len(vals)} values vs {n_groups} labels")

    n_fields = len(values_dict)
    width = 0.15  # width per bar
    colors = plt.get_cmap("Set1").colors

    # Centering offsets: symmetric offsets so group is centered
    offsets = (np.arange(n_fields) - (n_fields - 1) / 2.0) * width

    # Add gap between groups
    x = np.arange(n_groups) * (1 + group_gap)

    fig, ax = plt.subplots(figsize=(max(8 + n_groups * 0.2, 3), 3))

    for i, (y_field, vals) in enumerate(values_dict.items()):
        ax.bar(x + offsets[i], vals, width=width, label=y_field, color=colors[i % len(colors)])

    ax.set_xticks(x)
    ax.set_xticklabels(sim_start_labels, rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Simulation after iteration $i_s$")
    # ax.set_title(title)
    print(title)
    ax.grid(True, linestyle="--", alpha=0.5)

    # Optional zoom inset (vertical only)
    if zoom_window is not None:
        from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

        y_min, y_max = zoom_window  # zoom_window = (y_min, y_max)
        ax_inset = inset_axes(ax, width="95%", height="35%", loc="upper left",
                              bbox_to_anchor=(0.01, 0.7, 1, 0.8), bbox_transform=ax.transAxes,
                              borderpad=1)

        for i, (y_field, vals) in enumerate(values_dict.items()):
            ax_inset.bar(x + offsets[i], vals, width=width, color=colors[i % len(colors)])

        ax_inset.set_ylim(y_min, y_max)
        # keep full x-axis
        ax_inset.set_xlim(ax.get_xlim())

        ax_inset.grid(True, linestyle="--", alpha=0.5)
        ax_inset.set_xticks(x)
        ax_inset.set_xticklabels(sim_start_labels, rotation=0)
        mark_inset(ax, ax_inset, loc1=2, loc2=4, fc="none", ec="0.5")

    if y_limit is not None:
        ax.set_ylim(y_limit)

    # Legend
    if legend_outside:
        ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0., ncol=1)
    else:
        ax.legend(loc=legend_loc, ncol=1)

    if zoom_window is not None:
        plt.tight_layout(rect=[0, 0, 1.0, 1.0])  # Apply tight_layout to the main figure
    else:
        plt.tight_layout()
    set_axis_fontsize(ax)
    plt.show()
    save_figure(fig, filename, fmt="svg", bbox_inches='tight')
    plt.close(fig)


compute_accuracy_per_sim = True

# --- Prepare sims and fields ---
sims = sorted(data.keys())
y_fields_all = data[sims[-1]]["args"]["y_fields"]

# Map selected names to indices
if selected_y_fields is not None:
    selected_indices_fields = [i for i, f in enumerate(y_fields_all) if f in selected_y_fields]
else:
    selected_indices_fields = list(range(len(y_fields_all)))

# Keep only selected fields
y_fields = [y_fields_all[i] for i in selected_indices_fields]
n_y_fields = len(y_fields)
n_points_y = max(len(data[k]["y"]) for k in sims if data[k].get("y") is not None)
y = np.full((len(sims), n_points_y, n_y_fields), np.nan)
if start_iter is None:
    if len(sims) < 2:
        start_iter = [0, int(n_points_y / 4), int(n_points_y / 2), int(n_points_y / 4) + int(n_points_y / 2),
                      n_points_y - 1]
    else:
        start_iter = [0, int(n_points_y / 2), n_points_y - 1]

# --- Initialize arrays ---
y = np.full((len(sims), n_points_y, n_y_fields), np.nan)
y_pred = np.full_like(y, np.nan)
y_std = np.full_like(y, np.nan)

for sim_idx, k in enumerate(sims):
    if data[k].get("y") is not None:
        y_vals = np.array(data[k]["y"], dtype=float)
        y[sim_idx, :y_vals.shape[0], :] = y_vals[:, selected_indices_fields]

    if data[k].get("y_model_mean") is not None:
        y_pred_vals = np.array(data[k]["y_model_mean"], dtype=float)
        y_pred[sim_idx, :y_pred_vals.shape[0], :] = y_pred_vals[:, selected_indices_fields]

    if data[k].get("y_model_std") is not None:
        y_std_vals = np.array(data[k]["y_model_std"], dtype=float)
        y_std[sim_idx, :y_std_vals.shape[0], :] = y_std_vals[:, selected_indices_fields]

# --- Remove experts ---
if experts:
    sims = [s for s in sims if s not in experts]

# --- Select indices ---
plot_indices = selected_indices if selected_indices is not None else list(range(len(sims)))

# --- Colors ---
full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# --- Plot ---
if use_subfigures:
    n_rows = int(np.ceil(len(y_fields) / subplots_in_row))
    fig, axs = plt.subplots(n_rows, subplots_in_row, figsize=(figsize_single[0] * subplots_in_row,
                                                              figsize_single[1] * n_rows),
                            squeeze=False)
    axs = axs.flatten()
else:
    axs = [None] * len(y_fields)  # will create individual figs

for y_idx, y_field in enumerate(y_fields):
    ax = axs[y_idx] if use_subfigures else plt.subplots(figsize=figsize_single)[1]

    for plot_i, sim_idx in enumerate(plot_indices):
        sim_name = sims[sim_idx]

        mu = y_pred[sim_idx][:, y_idx]
        if np.all(np.isnan(mu)):  # --- Skip sims without predicted values ---
            continue

        actual = y[sim_idx][:, y_idx]
        sigma = y_std[sim_idx][:, y_idx]
        mask = ~np.isnan(actual) & ~np.isnan(mu)
        # mask = ~np.isnan(mu_all) & ~np.isnan(actual_all) & (~np.isnan(sigma_all) & (sigma_all > 0))
        if not np.any(mask):
            continue

        actual = actual[mask]
        mu = mu[mask]
        sigma = sigma[mask]

        # --- Normalize if requested ---
        if normalize:
            actual_normalized = actual / 100.0
            mu_normalized = mu / 100.0
            sigma_normalized = sigma / 100.0
        else:
            actual_normalized = actual
            mu_normalized = mu
            sigma_normalized = sigma

        # --- Flatten arrays ---
        actual_normalized = actual_normalized.flatten()
        mu_normalized = mu_normalized.flatten()
        sigma_normalized = sigma_normalized.flatten()

        # --- Ensure yerr is non-negative ---
        sigma_normalized = np.clip(sigma_normalized, 0, None)
        color = full_colors[plot_i % len(full_colors)]

        # --- Plot predicted ± std using errorbar ---
        ax.errorbar(
            x=mu_normalized,
            y=actual_normalized,
            xerr=sigma_normalized,
            fmt='o',
            color=color,
            ecolor=color,
            alpha=0.6,
            label=sim_name
        )
    # --- Compute min/max only if there are valid points ---
    if 'mu_normalized' in locals() and 'actual_normalized' in locals():
        all_points = np.concatenate([mu_normalized, actual_normalized])
        min_val = np.min(all_points)
        max_val = np.max(all_points)
    else:
        # skip this subplot if no valid data found
        ax.set_visible(False)
        continue

    # --- Reference line y=x ---
    # ax.plot([0, 1], [0, 1], "k--", lw=1.0)
    ax.plot([min_val, max_val], [min_val, max_val], "k--", lw=1.0)

    if zoom_around_points:
        # apply zoom margin
        range_val = max_val - min_val
        min_zoom = max(0.0, min_val - zoom_margin * range_val)
        max_zoom = min(1.0, max_val + zoom_margin * range_val)

        ax.set_xlim(min_zoom, max_zoom)
        ax.set_ylim(min_zoom, max_zoom)  # same as x-axis
    else:
        # ax.set_xlim(0, 1)
        # ax.set_ylim(0, 1)
        ax.set_xlim(min_val, max_val)
        ax.set_ylim(min_val, max_val)

    # --- Formatting ---
    norm = ""
    if normalize:
        norm = "(normalized)"
    ax.set_xlabel(f"{y_field} predicted {norm}", fontsize=12)
    ax.set_ylabel(f"{y_field} actual {norm}", fontsize=12)
    # ax.set_title(f"{y_field}: Predicted vs Actual", fontsize=13)
    ax.grid(True, linestyle="--", alpha=0.5)
    # ax.legend(loc="upper left", fontsize=9)
    ax.legend(loc="best", fontsize=9)
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.yaxis.set_major_locator(MaxNLocator(6))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

    set_axis_fontsize(ax)
    plt.tight_layout()
    if not use_subfigures:
        plt.show()
        save_figure(fig, f"{y_field}_consistnency_plot.svg", fmt="svg")

if use_subfigures:
    plt.show()
    save_figure(fig, "consistnency_plots.svg", fmt="svg")
if compute_accuracy_per_sim:
    import matplotlib.pyplot as plt
    import numpy as np
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    # --- Threshold for RMSE to decide GOOD/BAD ---
    rmse_threshold = 0.05  # adjust this based on your data scale

    # Ensure start_iter is a list
    if start_iter is None:
        start_iter_list = [0]  # default start at 0
    elif isinstance(start_iter, int):
        start_iter_list = [start_iter]
    elif isinstance(start_iter, (list, tuple)):
        start_iter_list = list(start_iter)
    else:
        raise TypeError("start_iter must be an int, list, tuple, or None")

    # --- Collect RMSE and Coverage per output and per start_iter for plotting ---
    rmse_plot = {y_field: [] for y_field in y_fields}
    coverage_plot = {y_field: [] for y_field in y_fields}
    wrmse_plot = {y_field: [] for y_field in y_fields}
    sim_start_labels = []

    for sim_idx in plot_indices:
        sim_name = sims[sim_idx]
        for start in start_iter_list:
            sim_start_labels.append(f"{sim_name}\n$i_s={start}$")
            for y_idx, y_field in enumerate(y_fields):
                mu_all = y_pred[sim_idx][start:, y_idx]
                actual_all = y[sim_idx][start:, y_idx]
                sigma_all = y_std[sim_idx][start:, y_idx]

                mask = ~np.isnan(mu_all) & ~np.isnan(actual_all) & (~np.isnan(sigma_all))
                # print(mask)
                # print(f"{y_field} | start={start} | mask sum={np.sum(mask)}")
                # print(f"mu_all={mu_all}")
                # print(f"actual_all={actual_all}")
                # print(f"sigma_all={sigma_all}")

                if np.sum(mask) == 0:
                    # No valid points, append NaN for all metrics
                    rmse_plot[y_field].append(np.nan)
                    coverage_plot[y_field].append(np.nan)
                    wrmse_plot[y_field].append(np.nan)
                    continue

                # Apply mask
                mu_sel = mu_all[mask]
                actual_sel = actual_all[mask]
                # sigma_sel = sigma_all[mask]
                sigma_sel = np.abs(sigma_all[mask])

                # Compute metrics
                mae = mean_absolute_error(actual_sel, mu_sel)
                rmse = np.sqrt(mean_squared_error(actual_sel, mu_sel))

                # Weighted RMSE formula (safe now)
                wrmse = np.sqrt(np.mean(((actual_sel - mu_sel) / sigma_sel) ** 2))

                if len(mu_sel) < 2:
                    r2 = np.nan
                else:
                    r2 = r2_score(actual_sel, mu_sel)

                coverage = np.mean((actual_sel >= mu_sel - sigma_sel) &
                                   (actual_sel <= mu_sel + sigma_sel)) * 100.0

                status = "GOOD" if (rmse <= rmse_threshold and coverage >= 66) else "BAD"

                print(f"{y_field} | {sim_name} | start_iter={start} | "
                      f"MAE: {mae:.3f}, RMSE: {rmse:.3f}, "
                      f"R²: {r2 if not np.isnan(r2) else 'nan'}, "
                      f"Coverage ±1σ: {coverage:.1f}% | {status}")

                # Append metrics
                rmse_plot[y_field].append(rmse)
                coverage_plot[y_field].append(coverage)
                wrmse_plot[y_field].append(wrmse)

    # --- Pad with NaNs for consistency ---
    for y_field in y_fields:
        if len(rmse_plot[y_field]) < len(sim_start_labels):
            rmse_plot[y_field].extend([np.nan] * (len(sim_start_labels) - len(rmse_plot[y_field])))
        if len(coverage_plot[y_field]) < len(sim_start_labels):
            coverage_plot[y_field].extend([np.nan] * (len(sim_start_labels) - len(coverage_plot[y_field])))

    # --- Use exactly the same layout for RMSE ---
    # Convert rmse_plot (dict of lists) to numpy lists (if not already)
    rmse_plot_clean = {yf: list(rmse_plot[yf]) for yf in y_fields}
    plot_grouped_bars(
        rmse_plot_clean,
        ylabel="RMSE",
        title="RMSE per Simulation, Output, and start_iter",
        filename="RMSE_centered.svg",
        # y_limit=[0, 3],
        y_limit=None,
        legend_loc="upper right",  # ignored if legend_outside=True
        legend_outside=True
    )
    plot_grouped_bars(
        rmse_plot_clean,
        ylabel="RMSE",
        title="RMSE per Simulation, Output, and start_iter",
        filename="RMSE_centered_zommed.svg",
        # y_limit=[0, 3],
        y_limit=None,
        legend_loc="upper right",  # ignored if legend_outside=True
        legend_outside=True,
        zoom_window=(0, 2)
    )
    # example: zoom first 6 groups, RMSE range 0–0.2

    # --- Plot WRMSE ---
    wrmse_plot_clean = {yf: list(wrmse_plot[yf]) for yf in y_fields}
    plot_grouped_bars(
        wrmse_plot_clean,
        ylabel="WRMSE (weighted by σ)",
        title="Weighted RMSE per Simulation, Output, and start_iter",
        filename="WRMSE_centered.svg",
        y_limit=None,
        legend_loc="upper right",
        legend_outside=True
    )

    # --- Coverage: same style, no threshold line, legend to the left ---
    coverage_plot_clean = {yf: list(coverage_plot[yf]) for yf in y_fields}
    plot_grouped_bars(
        coverage_plot_clean,
        ylabel="Coverage (%)",
        title="Coverage within ±1σ per Simulation, Output, and start_iter",
        filename="Coverage_centered.svg",
        y_limit=None,
        legend_loc="best",
        legend_outside=True
    )






In [ ]:
import re

# selected_indices = None
# Ensure selected_indices is defined
selected_indices = None


# selected_indices = [3]  # or None


def plot_over_iterations(field, name, sims=None, best_points=None, ax=None, save=False):
    show_fig = ax is None
    global selected_indices
    if sims is None:
        sims = sorted(data.keys())

    # Prepare sims and field according to selected_indices
    if selected_indices is not None:
        sims_to_plot = [sims[i] for i in selected_indices]
        field_to_plot = [field[i] for i in selected_indices]
        plot_indices = selected_indices
    else:
        sims_to_plot = sims
        field_to_plot = field
        plot_indices = list(range(len(sims)))

    full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
    colors = [full_colors[i % len(full_colors)] for i in plot_indices]
    markers = [all_markers[i % len(all_markers)] for i in plot_indices]

    if show_fig:
        fig, ax = plt.subplots(figsize=(8, 3))

    for plot_idx, sim_name in enumerate(sims_to_plot):
        vals = np.array(field_to_plot[plot_idx], dtype=float)
        mask = ~np.isnan(vals)
        if not np.any(mask):
            continue
        ax.plot(np.arange(len(vals))[mask], vals[mask],
                marker=markers[plot_idx], color=colors[plot_idx],
                alpha=0.8,
                linestyle="-", label=sim_name)

    if best_points:
        tab10 = plt.colormaps["tab10"]
        for rank, info in enumerate(best_points, 1):
            sim_name, iter_idx = info.get("sim"), info.get("iter")
            if sim_name not in sims_to_plot or iter_idx is None:
                continue
            sim_idx = sims_to_plot.index(sim_name)
            vals = np.array(field_to_plot[sim_idx], dtype=float)
            if iter_idx >= len(vals) or np.isnan(vals[iter_idx]):
                continue
            ax.scatter(
                iter_idx,
                vals[iter_idx],
                s=150,
                edgecolors="black",
                linewidths=2,
                marker=markers[sim_idx],
                color=tab10(rank / 10),
                zorder=6,
                label=f"Top {rank}"
            )
            info["rank"] = f"Top {rank}"

    ax.set_xlabel("iterations")
    ax.set_ylabel(name)
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(loc="upper right")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.tight_layout()
    set_axis_fontsize(ax)
    if show_fig:
        plt.show()
    if save:
        clean_name = re.sub(r"\(.*?\)", "", name).replace(" ", "")
        if best_points is not None:
            save_figure(fig, f"{clean_name}_vs_iterations_best_points.svg", fmt="svg")
        else:
            save_figure(fig, f"{clean_name}_vs_iterations.svg", fmt="svg")


def plot_cumulative_with_secondary(field, name, secondary_field=None, secondary_name=None, save=False):
    global selected_indices
    sims_all = sorted(data.keys())

    # Determine sims to plot and fields to plot
    if selected_indices is not None:
        sims_to_plot = [sims_all[i] for i in selected_indices]
        field_to_plot = [field[i] for i in selected_indices]
        if secondary_field is not None:
            secondary_to_plot = [secondary_field[i] for i in selected_indices]
        plot_indices = selected_indices
    else:
        sims_to_plot = sims_all
        field_to_plot = field
        secondary_to_plot = secondary_field
        plot_indices = list(range(len(sims_all)))

    fig, ax_right = plt.subplots(figsize=(12, 3) if secondary_field is not None else (8, 3))
    full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
    colors = [full_colors[i % len(full_colors)] for i in plot_indices]
    markers = [all_markers[i % len(all_markers)] for i in plot_indices]

    # --- main cumulative field ---
    for plot_idx, sim_name in enumerate(sims_to_plot):
        vals = np.array(field_to_plot[plot_idx], dtype=float)
        if not np.any(~np.isnan(vals)):
            continue
        cum_vals = np.nancumsum(vals)
        iters = np.arange(len(cum_vals))
        ax_right.plot(iters, cum_vals, marker=markers[plot_idx], color=colors[plot_idx],
                      alpha=0.8, linestyle="-",
                      label=f"{sim_name} ({name})" if secondary_field is not None else sim_name)
        ax_right.fill_between(iters, 0, cum_vals, alpha=0.2, color=colors[plot_idx])

    ax_right.set_xlabel("iterations")
    ax_right.set_ylabel(name)
    ax_right.grid(True, linestyle="--", alpha=0.6)
    ax_right.xaxis.set_major_locator(MaxNLocator(integer=True))
    set_axis_fontsize(ax_right)

    # --- secondary field ---
    if secondary_field is not None:
        ax_left = ax_right.twinx()
        for plot_idx, sim_name in enumerate(sims_to_plot):
            sec_vals = np.array(secondary_to_plot[plot_idx], dtype=float)
            if not np.any(~np.isnan(sec_vals)):
                continue
            cum_sec = np.nancumsum(sec_vals)
            ax_left.plot(iters, cum_sec, marker=markers[plot_idx], color=colors[plot_idx],
                         linestyle=":", label=f"{sim_name} ({secondary_name})")
            ax_left.fill_between(iters, 0, cum_sec, alpha=0.1, color=colors[plot_idx])
        ax_left.set_ylabel(secondary_name)

        # combined legend
        lines_r, labels_r = ax_right.get_legend_handles_labels()
        lines_l, labels_l = ax_left.get_legend_handles_labels()
        ax_right.legend(lines_r + lines_l, labels_r + labels_l, loc="center left",
                        bbox_to_anchor=(1.1, 0.6))
    else:
        ax_right.legend(loc="best")

    plt.tight_layout()
    set_axis_fontsize(ax_right)
    plt.show()
    if save:
        clean_name = re.sub(r"\(.*?\)", "", name).replace(" ", "")
        if secondary_field is not None:
            clean_secondary = re.sub(r"\(.*?\)", "", secondary_name).replace(" ", "")
            save_figure(fig, f"cumulative_{clean_name.lower()}_and_{clean_secondary.lower()}_vs_iterations.svg",
                        fmt="svg")
        else:
            save_figure(fig, f"cumulative_{clean_name.lower()}_vs_iterations.svg", fmt="svg")


# --- Data setup and plotting ---
sims = sorted(data.keys())
energy = [data[k].get("energy") or data[k].get("sim_energy", None) for k in data]
all_energy = np.concatenate([np.array(e).ravel() for e in energy if e is not None])
max_energy = np.max(all_energy)
factor, unit = get_unit(max_energy, "J")
co2 = [data[k].get("co2", None) for k in data]
train_energy = [data[k].get("train_energy", None) for k in data]
if len(train_energy) > 0:
    all_train_energy = np.concatenate([np.array(e).ravel() for e in energy if e is not None])
    max_train_energy = np.max(all_train_energy)
    factor_train, unit_train = get_unit(max_train_energy, "J")

elapsed_seconds = [data[k].get("elapsed_seconds", None) for k in data]

# --- Add experts ---
if experts:
    for name, sim in experts.items():
        sims.append(name)
        expert_energy = np.array(sim.get("energy", []), dtype=float)
        expert_elapsed = np.array(sim.get("elapsed_seconds", []), dtype=float)
        expert_co2 = np.array(sim.get("co2", []), dtype=float)

        max_len = max([len(e) for e in energy] + [len(expert_energy)])
        if len(expert_energy) < max_len:
            pad = max_len - len(expert_energy)
            expert_energy = np.pad(expert_energy, (0, pad), constant_values=np.nan)
            expert_elapsed = np.pad(expert_elapsed, (0, pad), constant_values=np.nan)
            expert_co2 = np.pad(expert_co2, (0, pad), constant_values=np.nan)

        energy.append(expert_energy)
        elapsed_seconds.append(expert_elapsed)
        co2.append(expert_co2)

energy = np.array(energy) / factor
elapsed_minutes = [[v / 60 for v in sim] for sim in elapsed_seconds]
if len(train_energy) > 0 and len([v for v in train_energy if v is not None]) > 0:
    train_energy = np.array(train_energy) / factor_train

# --- Plot all ---
if len(train_energy) > 0 and len([v for v in train_energy if v is not None]) > 0:
    plot_over_iterations(train_energy, f"Train Energy ({unit_train})", sims=sorted(data.keys()), save=True)

plot_over_iterations(energy, f"Energy ({unit})", best_points=best_points, sims=sims, save=True)
plot_over_iterations(energy, f"Energy ({unit})", sims=sims, save=True)
plot_over_iterations(co2, "CO2 (g)", sims=sims)
plot_over_iterations(elapsed_minutes, "Time (minutes)", sims=sims, save=True)

plot_cumulative_with_secondary(elapsed_seconds, "Time (seconds)", save=True)
plot_cumulative_with_secondary(energy, f"Energy ({unit})", save=True)
# ax = plt.gca()
# ax.set_xlim(0,3)
plot_cumulative_with_secondary(elapsed_seconds, "Time (seconds)", energy, f"Energy ({unit})", save=True)


In [ ]:
# selected_indices = [0, 2, 4]
selected_indices = None

# walltime
sims = sorted(data.keys())

import re

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset


def plot_comm_steps(field, name, secondary_field=None, secondary_name=None,
                    parallel_samples=None, method=None, save: bool = False, zoom_window=None):
    # Determine simulations to plot
    sims_to_plot = [sims[i] for i in selected_indices] if selected_indices is not None else sims
    plot_indices = selected_indices if selected_indices is not None else list(range(len(sims)))

    # Colors and markers mapped to plot_indices
    full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
    colors = [full_colors[i % len(full_colors)] for i in plot_indices]
    markers = [all_markers[i % len(all_markers)] for i in plot_indices]

    # --- Setup figure ---
    fig, ax_right = plt.subplots(figsize=(8.7, 3))

    lines, labels = [], []

    # --- main field ---
    for plot_idx, sim_name in enumerate(sims_to_plot):
        # Use field data safely
        if selected_indices is not None:
            vals = np.array(field[selected_indices[plot_idx]])
        else:
            vals = np.array(field[plot_idx])

        if not np.any(~np.isnan(vals)):
            continue
        # Remove nan values
        mask = ~np.isnan(vals)
        vals_clean = vals[mask]
        iterations = np.arange(len(vals_clean))

        if len(vals_clean) == 0:
            continue

        cumulative_vals = np.nancumsum(vals_clean)
        ps = f"{parallel_samples[plot_idx]} ps" if parallel_samples else ""
        m = f", {method[plot_idx]}" if method and parallel_samples and parallel_samples[plot_idx] > 1 else ""
        label = f"{sim_name} ({ps}{m}) " if "sim" in sim_name else f"{sim_name}"
        l, = ax_right.plot(iterations, cumulative_vals,
                           marker=markers[plot_idx], linestyle="-", color=colors[plot_idx],
                           alpha=0.8,
                           label=label)
        ax_right.fill_between(iterations, 0, cumulative_vals, alpha=0.2, color=colors[plot_idx])
        lines.append(l)
        labels.append(label)

    ax_right.set_xlabel("Cycles")
    ax_right.set_ylabel(name)
    ax_right.grid(True, linestyle="--", alpha=0.6)
    ax_right.xaxis.set_major_locator(MaxNLocator(integer=True))
    set_axis_fontsize(ax_right)
    # ax_right.set_ylim(0, 400)

    # --- Optional zoom inset ---
    if zoom_window:
        fig = ax_right.figure
        fig.set_size_inches(8.7, 3.5, forward=True)  # width=10, height=4 inches
        x_min, x_max, y_min, y_max = zoom_window
        ax_inset = inset_axes(ax_right, width="35%", height="35%", loc="upper left",
                              bbox_to_anchor=(0.15, -.4, 1, 1.7), bbox_transform=ax_right.transAxes,
                              # bbox_to_anchor=(0.05, -.5, 1, 1.7), bbox_transform=ax_right.transAxes,
                              borderpad=1)

        for plot_idx, sim_name in enumerate(sims_to_plot):
            vals = np.array(field[plot_idx] if selected_indices is None else field[selected_indices[plot_idx]])
            mask = ~np.isnan(vals)
            vals_clean = vals[mask]
            iterations = np.arange(len(vals_clean))
            cumulative_vals = np.nancumsum(vals_clean)
            ax_inset.plot(iterations, cumulative_vals, marker=markers[plot_idx], linestyle="-", color=colors[plot_idx])
        ax_inset.set_xlim(x_min, x_max)
        ax_inset.set_ylim(y_min, y_max)
        ax_inset.grid(True, linestyle="--", alpha=0.5)
        mark_inset(ax_right, ax_inset, loc1=2, loc2=4, fc="none", ec="0.5")

    # --- secondary field ---
    if secondary_field is not None and secondary_name is not None:
        ax_left = ax_right.twinx()
        for plot_idx, sim_name in enumerate(sims_to_plot):
            if selected_indices is not None:
                sec_vals = np.array(secondary_field[selected_indices[plot_idx]], dtype=float)
            else:
                sec_vals = np.array(secondary_field[plot_idx], dtype=float)

            if not np.any(~np.isnan(sec_vals)):
                continue

            # Remove nan values
            mask = ~np.isnan(sec_vals)
            sec_vals_clean = sec_vals[mask]
            iterations = np.arange(len(sec_vals_clean))

            if len(sec_vals_clean) == 0:
                continue

            sec_cum = np.nancumsum(sec_vals_clean)
            ps = f"{parallel_samples[plot_idx]} ps" if parallel_samples else ""
            m = f", {method[plot_idx]} " if method and parallel_samples and parallel_samples[plot_idx] > 1 else ""
            label = f"{sim_name} ({ps}{m})"
            l, = ax_left.plot(iterations, sec_cum,
                              marker=markers[plot_idx], linestyle=":", color=colors[plot_idx],
                              alpha=0.8,
                              label=label)
            ax_left.fill_between(iterations, 0, sec_cum, alpha=0.1, color=colors[plot_idx])
            lines.append(l)
            labels.append(label)
        ax_left.set_ylabel(secondary_name)
        set_axis_fontsize(ax_left)

        ax_right.legend(lines, labels, loc="center left", bbox_to_anchor=(1.15, 0.62))
    else:
        ax_right.legend(lines, labels, loc="lower right")

    if zoom_window is not None:
        plt.tight_layout(rect=[0, 0, 1.0, 1.0])  # Apply tight_layout to the main figure
    else:
        plt.tight_layout()

    if save:
        first_name = re.sub(r"\(.*?\)", "", name).replace(" ", "")
        save_name = f"cumulative_cycles_{first_name.lower()}"
        if secondary_field is not None and secondary_name is not None:
            secondary_name = re.sub(r"\(.*?\)", "", secondary_name).replace(" ", "")
            save_name += f"_and_{secondary_name.lower()}_vs_iterations"
        else:
            save_name += "_vs_iterations"
        if zoom_window is not None:
            save_name += "_zoomed"
        save_figure(fig, save_name + ".svg", fmt="svg", bbox_inches='tight')


def merge_parallel(samples_list, parallel_samples_list):
    """
    Accumulate consecutive entries according to parallel_samples (sum).

    Args:
        samples_list: list of lists, e.g., elapsed_minutes per simulation
        parallel_samples_list: list of int, number of parallel samples per sim

    Returns:
        merged_list: list of lists with accumulated values, padded with None if needed
    """
    merged_list = []
    for sim_samples, parallel in zip(samples_list, parallel_samples_list):
        if parallel == 1:
            merged_list.append(list(sim_samples))  # no merge needed
            continue

        merged_sim = []
        for i in range(0, len(sim_samples), parallel):
            chunk = sim_samples[i:i + parallel]
            if len(chunk) < parallel:
                chunk += [None] * (parallel - len(chunk))
            merged_value = sum(v for v in chunk if v is not None)
            merged_sim.append(merged_value)
        merged_list.append(merged_sim)
    return merged_list


def merge_parallel_max(samples_list, parallel_samples_list):
    """
    Take the max of consecutive entries according to parallel_samples.

    Args:
        samples_list: list of lists, e.g., elapsed_minutes per simulation
        parallel_samples_list: list of int, number of parallel samples per sim

    Returns:
        merged_list: list of lists with max values, padded with None if needed
    """
    merged_list = []
    for sim_samples, parallel in zip(samples_list, parallel_samples_list):
        if parallel == 1:
            merged_list.append(list(sim_samples))  # no merge needed
            continue

        merged_sim = []
        for i in range(0, len(sim_samples), parallel):
            chunk = sim_samples[i:i + parallel]
            if len(chunk) < parallel:
                chunk += [None] * (parallel - len(chunk))
            valid_values = [v for v in chunk if v is not None]
            merged_value = max(valid_values) if valid_values else None
            merged_sim.append(merged_value)
        merged_list.append(merged_sim)
    return merged_list


# parallel samples are executed together
parallel_samples = [data[k].get("args")["parallel_samples"] for k in data]
method = [data[k].get("args")["selection_strategy"] for k in data]
print(parallel_samples)
elapsed_seconds = [data[k].get("elapsed_seconds", None) for k in data]
elapsed_minutes = [
    [v / 60 for v in sim] for sim in elapsed_seconds
]
# compute max time over the itterations
merged_minutes = merge_parallel_max(elapsed_minutes, parallel_samples)

energy = [data[k].get("energy") or data[k].get("sim_energy", None) for k in data]
all_energy = np.concatenate([np.array(e).ravel() for e in energy if e is not None])
max_energy = np.max(all_energy)
factor, unit = get_unit(max_energy, "J")
# make sure parallel_samples is updated for experts
if experts:
    for name, sim in experts.items():
        sims.append(name)

        # elapsed_seconds (minutes)
        expert_minutes = np.array(sim.get("elapsed_seconds", []), dtype=float) / 60.0
        max_len = max([len(e) for e in merged_minutes] + [len(expert_minutes)])
        if len(expert_minutes) < max_len:
            expert_minutes = np.pad(expert_minutes, (0, max_len - len(expert_minutes)), constant_values=np.nan)
        merged_minutes.append(expert_minutes)

        # energy
        expert_energy = np.array(sim.get("energy", []), dtype=float)
        max_len = max([len(e) for e in energy if e is not None] + [len(expert_energy)])
        if len(expert_energy) < max_len:
            expert_energy = np.pad(expert_energy, (0, max_len - len(expert_energy)), constant_values=np.nan)
        energy.append(expert_energy)

        # co2
        expert_co2 = np.array(sim.get("co2", []), dtype=float)
        max_len = max([len(e) for e in co2 if e is not None] + [len(expert_co2)])
        if len(expert_co2) < max_len:
            expert_co2 = np.pad(expert_co2, (0, max_len - len(expert_co2)), constant_values=np.nan)
        co2.append(expert_co2)

        # important: keep parallel_samples aligned
        parallel_samples.append(1)  # or whatever number of parallel runs this expert had
        method.append("")

energy = np.array(energy) / factor
energy = list(energy)
merged_energy = merge_parallel(energy, parallel_samples)
# print(merged_energy)


plot_comm_steps(merged_minutes, "Walltime (Minutes)", parallel_samples=parallel_samples, save=True)

zoom_window = (1, 4, 180, 500)
special = {"crowding_distance", "peak_based", "highest_sum"}
if len(special.intersection(method)) >= 2:
    plot_comm_steps(merged_energy, f"Energy ({unit})", parallel_samples=parallel_samples, method=method, save=True)
    plot_comm_steps(merged_energy, f"Energy ({unit})", parallel_samples=parallel_samples, method=method, save=True,
                    zoom_window=zoom_window)
else:
    plot_comm_steps(merged_energy, f"Energy ({unit})", parallel_samples=parallel_samples, save=True,
                    zoom_window=zoom_window)
    plot_comm_steps(merged_energy, f"Energy ({unit})", parallel_samples=parallel_samples, save=True)

plot_comm_steps(merged_minutes, "Walltime (Minutes)", merged_energy, f"Energy ({unit})",
                parallel_samples=parallel_samples,
                save=True)

if "DACH-VWS/DACH-VWS" in json_file:
    entries = np.array(merged_minutes[3], dtype=float)
    user_sum = np.nansum(entries)  # automatically ignores NaN
    entries = np.array(merged_minutes[4], dtype=float)
    expert_sum = np.nansum(entries)  # automatically ignores NaN
    print(
        f"total time{user_sum} minutes.\n Speedup: {user_sum / merged_minutes[0][0]}x {user_sum / merged_minutes[1][0]}x {user_sum / merged_minutes[2][0]}x")
    print(
        f"total time{expert_sum} minutes.\n Speedup: {expert_sum / merged_minutes[0][0]}x {expert_sum / merged_minutes[1][0]}x {expert_sum / merged_minutes[2][0]}x")

In [ ]:
sims = sorted(data.keys())
if "/DACH-VWS/" in json_file and len(sims) > 5 or "SeatShell_168_highestpeak_added" in json_file:
    selected_indices = [2, 4]
else:
    selected_indices = None

# --- Determine sims to plot ---
sims_to_plot = [sims[i] for i in selected_indices] if selected_indices is not None else sims
plot_indices = selected_indices if selected_indices is not None else list(range(len(sims)))
progress = [data[k].get("progress") for k in sims]

# --- Colors, markers, linestyles ---
full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
colors = [full_colors[i % len(full_colors)] for i in plot_indices]
markers = [all_markers[i % len(all_markers)] for i in plot_indices]
solid_ls = ["-"] * len(sims_to_plot)
dash_ls = ["--"] * len(sims_to_plot)

# --- Progress plot (single axis, solid lines) ---
fig, ax = plt.subplots(figsize=(8.7, 3))
for plot_idx, sim_name in enumerate(sims_to_plot):
    iterations = list(range(len(progress[plot_idx])))
    ax.plot(
        iterations,
        progress[plot_idx],
        color=colors[plot_idx],
        alpha=0.8,
        marker=markers[plot_idx],
        linestyle=solid_ls[plot_idx],
        label=f"{sim_name}"
    )

ax.set_xlabel("iterations")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_ylabel("Progress (%)")
set_axis_fontsize(ax)
ax.legend(loc="lower left")
ax.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()
save_figure(fig, "process_vs_iterations.svg", fmt="svg")

# Prepare energy
energy = [data[k].get("energy") or data[k].get("sim_energy", None) for k in sims]
all_energy = np.concatenate([np.array(e).ravel() for e in energy if e is not None])
max_energy = np.max(all_energy)
factor, unit = get_unit(max_energy, "J")

# Include experts
if experts:
    for name, sim in experts.items():
        sims.append(name)
        expert_energy = np.array(sim.get("energy", []), dtype=float)
        max_len = max(len(e) for e in energy if e is not None)
        if len(expert_energy) < max_len:
            expert_energy = np.pad(expert_energy, (0, max_len - len(expert_energy)), constant_values=np.nan)
        energy.append(expert_energy)

sims = sorted(data.keys())
if selected_indices is not None:
    sims_to_plot = [sims[i] for i in selected_indices]
else:
    sims_to_plot = sims

# --- Color and marker setup ---
full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
if selected_indices is not None:
    colors = [full_colors[i % len(full_colors)] for i in selected_indices]
    markers = [all_markers[i % len(all_markers)] for i in selected_indices]
else:
    colors = [full_colors[i % len(full_colors)] for i in range(len(sims))]
    markers = all_markers[:len(sims)]

selected_indices = [0, 2, 4]
# --- Plot energy over iterations ---
fig, ax = plt.subplots(figsize=(8.7, 3))
for plot_idx, sim_name in enumerate(sims_to_plot):
    sim_idx = sims.index(sim_name)
    y_energy = np.array(energy[sim_idx]) / factor
    mask = ~np.isnan(y_energy)
    if not np.any(mask):
        continue
    iterations = np.arange(len(y_energy))[mask]
    plt.plot(iterations, np.array(y_energy)[mask],
             marker=markers[plot_idx], color=colors[plot_idx],
             alpha=0.8,
             linestyle="-", label=sim_name)

plt.xlabel("iterations")
plt.ylabel(f"Energy ({unit})")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
set_axis_fontsize(ax)
plt.show()
save_figure(fig, "energy_selected_vs_iterations.svg", fmt="svg")

fig, ax1 = plt.subplots(figsize=(11, 4))
ax2 = ax1.twinx()

for plot_idx, sim_name in enumerate(sims_to_plot):
    iterations = list(range(len(energy[plot_idx])))
    ax1.plot(iterations, progress[plot_idx], marker=markers[plot_idx],
             linestyle=solid_ls[plot_idx], color=colors[plot_idx],
             alpha=0.8,
             label=f"{sim_name} (progress)")
    ax2.plot(iterations, np.array(energy[plot_idx]) / factor, marker=markers[plot_idx],
             linestyle=dash_ls[plot_idx], color=colors[plot_idx],
             alpha=0.8,
             label=f"{sim_name} (energy)")

ax1.set_xlabel("iterations")
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
ax1.set_ylabel("Progress (%)", color="tab:cyan")
ax2.set_ylabel(f"Energy ({unit})", color="tab:red")
set_axis_fontsize(ax1)
ax1.grid(True, linestyle="--", alpha=0.6)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc="center left", bbox_to_anchor=(1.1, 0.85))

plt.tight_layout()
plt.show()
save_figure(fig, "process_and_energy_vs_iterations.svg", fmt="svg")


In [ ]:
if "/DACH-VWS/" in json_file and len(sims) > 5:
    selected_indices = [0, 3, 4]
else:
    selected_indices = None
# sims = sorted(data.keys())

# --- Determine sims to plot ---
sims_to_plot = [sims[i] for i in selected_indices] if selected_indices is not None else sims
plot_indices = selected_indices if selected_indices is not None else list(range(len(sims)))
progress = [data[k].get("progress") for k in data]

# --- Colors, markers, linestyles ---
full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
colors = [full_colors[i % len(full_colors)] for i in plot_indices]
markers = [all_markers[i % len(all_markers)] for i in plot_indices]
solid_ls = ["-"] * len(sims_to_plot)
dash_ls = ["--"] * len(sims_to_plot)

# --- Y-fields plots (single axis, solid lines) ---
y = np.array([data[k].get("y", None) for k in sims_to_plot], dtype=object)
y_fields = data[sims_to_plot[-1]]["args"]["y_fields"]

for y_idx, y_field in enumerate(y_fields):
    fig, ax = plt.subplots(figsize=(8, 3))

    for plot_idx, sim_name in enumerate(sims_to_plot):
        y_vals = y[plot_idx][:, y_idx].astype(float)
        iterations = list(range(len(y_vals)))
        ax.plot(iterations, y_vals,
                marker=markers[plot_idx],
                linestyle=solid_ls[plot_idx],
                color=colors[plot_idx],
                alpha=0.8,
                label=f"{sim_name}")

    # --- Highlight top points safely ---
    if best_points:
        tab10 = plt.colormaps["tab10"]  # ListedColormap
        for rank, info in enumerate(best_points, 1):
            sim_name = info.get("sim")
            iter_idx = info.get("iter")
            if sim_name not in sims_to_plot or iter_idx is None:
                continue
            sim_idx = sims_to_plot.index(sim_name)
            y_val = float(y[sim_idx][iter_idx, y_idx])
            if np.isnan(y_val):
                continue
            ax.scatter(
                iter_idx,
                y_val,
                s=120,
                edgecolors="black",
                linewidths=1.5,
                color=tab10(rank),  # normalize rank to [0,1]
                marker=markers[sim_idx],
                zorder=5,
                label=f"Top {rank}"
            )
            info["rank"] = f"Top {rank}"

    ax.set_xlabel("iterations")
    ax.set_ylabel(f"{y_field} (%)")
    ax.set_title(f"{y_field} vs Iterations")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend()
    set_axis_fontsize(ax)
    plt.tight_layout()
    plt.show()
    save_figure(fig, f"{y_field}_vs_iterations_best_point.svg", fmt="svg")


In [ ]:
if "/DACH-VWS/" in json_file and len(sims) > 5:
    selected_indices = [0, 2, 4]
else:
    selected_indices = None

sims_all = sorted(data.keys())
sims = [sims_all[i] for i in selected_indices] if selected_indices is not None else sims_all

# --- Prepare data ---
y = np.array([data[k].get("y", None) for k in sims], dtype=object)
y_fields = data[sims_all[-1]]["args"]["y_fields"]  # use global ordering

# --- Field indices ---
left_idx = 6
right_idx = 3

# --- Color and marker setup ---
full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
n_sims = len(sims)
if selected_indices is not None:
    colors = [full_colors[i % len(full_colors)] for i in selected_indices]
    markers = [all_markers[i % len(all_markers)] for i in selected_indices]
else:
    colors = [full_colors[i % len(full_colors)] for i in range(n_sims)]
    markers = [all_markers[i % len(all_markers)] for i in range(n_sims)]

solid_ls = ["-"] * n_sims
dash_ls = ["--"] * n_sims

# --- Create figure and left axis (solid) ---
fig, ax_left = plt.subplots(figsize=(8, 3))
for plot_idx in range(n_sims):
    y_vals = y[plot_idx][:, left_idx].astype(float)
    iterations = np.arange(len(y_vals))
    ax_left.plot(
        iterations, y_vals,
        color=colors[plot_idx],
        alpha=0.8,
        marker=markers[plot_idx],
        linestyle=solid_ls[plot_idx],
        label=f"{sims[plot_idx]} ({y_fields[left_idx]})"
    )

ax_left.set_xlabel("iterations")
ax_left.set_ylabel(y_fields[left_idx])
ax_left.grid(True, linestyle="--", alpha=0.6)
ax_left.xaxis.set_major_locator(MaxNLocator(integer=True))

# --- Create right axis (dashed) ---
ax_right = ax_left.twinx()
for plot_idx in range(n_sims):
    y_vals = y[plot_idx][:, right_idx].astype(float)
    iterations = np.arange(len(y_vals))
    ax_right.plot(
        iterations, y_vals,
        color=colors[plot_idx],  # same color as left axis
        alpha=0.8,
        marker=markers[plot_idx],  # same marker
        linestyle=dash_ls[plot_idx],  # dashed for right axis
        label=f"{sims[plot_idx]} ({y_fields[right_idx]})"
    )

ax_right.set_ylabel(y_fields[right_idx])
ax_left.xaxis.set_major_locator(MaxNLocator(integer=True))

# --- Combine legends ---
lines_left, labels_left = ax_left.get_legend_handles_labels()
lines_right, labels_right = ax_right.get_legend_handles_labels()
fig.legend(
    lines_left + lines_right,
    labels_left + labels_right,
    loc="upper right",
    bbox_to_anchor=(1.14, 0.9)
)

# --- Final layout ---
set_axis_fontsize(ax_left)
plt.tight_layout()
plt.show()
save_figure(fig, f"{y_fields[left_idx]}_and_{y_fields[right_idx]}_vs_iterations.svg", fmt="svg")

# --- Optional: plot EI ---
ei = [data[k].get("ei_sum", None) for k in sims]
plot_over_iterations(ei, "EI Sum", save=True)


To find the best y, we need to multiply by the approximated coefficient and apply the same condition as in the code
This snipped finds the best point

In [ ]:
if "/DACH-VWS/" in json_file and len(sims) > 5:
    selected_indices = [0, 2, 4]
else:
    selected_indices = None

sims = sorted(data.keys())
if selected_indices is not None:
    sims_to_plot = [sims[i] for i in selected_indices]
else:
    sims_to_plot = sims

# Prepare energy
energy = [data[k].get("energy") or data[k].get("sim_energy", None) for k in sims]

# Include experts
if experts:
    for name, sim in experts.items():
        sims.append(name)
        expert_energy = np.array(sim.get("energy", []), dtype=float)
        max_len = max(len(e) for e in energy if e is not None)
        if len(expert_energy) < max_len:
            expert_energy = np.pad(expert_energy, (0, max_len - len(expert_energy)), constant_values=np.nan)
        energy.append(expert_energy)

# --- Color and marker setup ---
full_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
all_markers = ["o", "s", "v", "^", "d", "x", "*", "P", "H", "+"]
if selected_indices is not None:
    colors = [full_colors[i % len(full_colors)] for i in selected_indices]
    markers = [all_markers[i % len(all_markers)] for i in selected_indices]
else:
    colors = [full_colors[i % len(full_colors)] for i in range(len(sims))]
    markers = all_markers[:len(sims)]

# --- Plot energy over iterations ---
plt.figure(figsize=(8, 3))
for plot_idx, sim_name in enumerate(sims_to_plot):
    sim_idx = sims.index(sim_name)
    y_energy = energy[sim_idx]
    mask = ~np.isnan(y_energy)
    if not np.any(mask):
        continue
    iterations = np.arange(len(y_energy))[mask]
    plt.plot(iterations, np.array(y_energy)[mask],
             marker=markers[plot_idx], color=colors[plot_idx],
             alpha=0.8,
             linestyle="-", label=sim_name)

# Use modern Matplotlib colormap

# Highlight best points safely
if best_points:
    tab10 = plt.colormaps["tab10"]
    for rank, info in enumerate(best_points, 1):
        sim_name = info.get("sim")
        if sim_name not in sims_to_plot:
            continue
        iter_idx = info.get("iter")
        energy_val = info.get("energy")
        if iter_idx is None or energy_val is None:
            continue
        sim_plot_idx = sims_to_plot.index(sim_name)

        # Use original sim index to map color
        if sim_name in sims_to_plot:
            orig_idx = sims_to_plot.index(sim_name)
        else:
            orig_idx = len(sims_to_plot)  # fallback for experts

        plt.scatter(
            iter_idx,
            energy_val,
            s=120,
            edgecolors="black",
            color=tab10(rank),  # color matches simulation line
            marker=markers[sim_plot_idx],  # marker matches simulation line
            zorder=5,
            label=f"Top {rank}"
        )
        info["rank"] = f"Top {rank}"

plt.xlabel("iterations")
plt.ylabel("Energy")
# plt.title("Energy per Simulation (best points highlighted)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()
save_figure(fig, "energy)and_best_points__vs_iterations.svg", fmt="svg")

# --- Plot energy per simulation as vertical lines ---
plt.figure(figsize=(8, 3))
x_indices = np.arange(len(sims_to_plot))
for plot_idx, sim_name in enumerate(sims_to_plot):
    sim_idx = sims.index(sim_name)
    y_energy = energy[sim_idx]
    mask = ~np.isnan(y_energy)
    if not np.any(mask):
        continue
    x_sim = np.full_like(np.array(y_energy)[mask], x_indices[plot_idx], dtype=float)
    plt.plot(x_sim, np.array(y_energy)[mask],
             marker=markers[plot_idx], color=colors[plot_idx],
             alpha=0.8,
             linestyle="-", label=sim_name)

# Use tab10 colormap (modern Matplotlib)
tab10 = plt.colormaps["tab10"]

# Highlight top points
if best_points:
    for rank, info in enumerate(best_points, 1):
        sim_name = info.get("sim")
        if sim_name not in sims_to_plot:
            continue
        iter_idx = info.get("iter")
        energy_val = info.get("energy")
        if iter_idx is None or energy_val is None:
            continue
        sim_plot_idx = sims_to_plot.index(sim_name)

        # color based on the original sim index to match plotted line
        if sim_name in sims_to_plot:
            orig_idx = sims_to_plot.index(sim_name)
        else:  # fallback for experts
            orig_idx = len(sims_to_plot)

        plt.scatter(
            x_indices[sim_plot_idx],
            energy_val,
            s=120,
            edgecolors="black",
            marker=markers[sim_plot_idx],
            color=tab10(rank),  # scaled to [0,1]
            zorder=5,
            label=f"Top {rank}"
        )

plt.xticks(ticks=x_indices, labels=sims_to_plot)
plt.xlabel("Simulation")
plt.ylabel("Energy")
plt.title("Energy per Simulation")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
set_axis_fontsize(plt.gca())
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

sims = sorted(data.keys())
# Get all args dicts
args_list = [data[k]["args"] for k in sims]

# Collect all keys
all_keys = set().union(*[args.keys() for args in args_list])

# Build table
table = []
for key in all_keys:
    values = [args.get(key, None) for args in args_list]
    # Check if all values are equal
    all_equal = all(v == values[0] for v in values)
    table.append({
        "Field": key,
        "Values": values,
        "Equal Across Sims": all_equal
    })

df = pd.DataFrame(table)
print(df)

# Filter rows where values are not equal
diff_fields = df[~df["Equal Across Sims"]]
# Print the differing fields




In [ ]:
cols = ["Field", "Values"]
print(f"\n\n{diff_fields[cols]}")

In [ ]:


# Show all rows and columns without truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
cols = ["Field", "Values"]
print(f"\n\n{diff_fields[cols]}")
print(sims)



In [ ]:
# print(args_list[-1])

In [ ]:
row = df[df["Field"] == "selection_strategy"]
print(row)